# Building a Data Pipeline with dlt

In this notebook, we build a complete data pipeline from scratch using **dlt**.

Goal:

- Fetch real data from an API
- Normalize nested JSON into relational tables
- Load data into a database (DuckDB)
- Explore and analyze the loaded data

You will understand the dlt flow step-by-step: **Extract → Normalize → Load**.


## 📦 Step 0: Install Dependencies


In [1]:
# install dependencies first
!pip -q install "dlt[duckdb]"


zsh:1: no matches found: dlt[duckdb]


<p>In this notebook we use:</p>

<ul>
  <li><strong>dlt</strong> for extract, normalize, and load</li>
  <li><strong>DuckDB</strong> as the local destination database</li>
</ul>

<p>DuckDB is beginner-friendly because it runs locally without extra setup.</p>


## 📚 Step 1: Import Libraries


<p>Import the libraries used throughout this notebook:</p>

<ul>
  <li><strong>dlt</strong>: pipeline engine</li>
  <li><strong>rest_api_source</strong>: helper for REST API source config</li>
  <li><strong>islice</strong>: helper to preview a few records</li>
</ul>


In [2]:
import dlt
from itertools import islice
from dlt.sources.rest_api import rest_api_source


## 🔗 Step 2: Define the API Source (Open Library)


<p>
In dlt, a <strong>source</strong> defines where data comes from and how to fetch it.
Here we use the <strong>Open Library Search API</strong>.
</p>

<p>
<code>rest_api_source</code> lets you define request and pagination behavior with a simple dictionary.
</p>

<p>
📖 API docs:
<a href="https://openlibrary.org/dev/docs/api/search" target="_blank">https://openlibrary.org/dev/docs/api/search</a>
</p>


In [3]:
def openlibrary_source(query: str = "harry potter"):

    return rest_api_source({
        "client": {
            "base_url": "https://openlibrary.org",
        },
        "resource_defaults": {
            "primary_key": "key",
            "write_disposition": "replace",
        },
        "resources": [
            {
                "name": "books",
                "endpoint": {
                    "path": "search.json",
                    "params": {
                        "q": query,
                        "limit": 100,
                    },
                    "data_selector": "docs",
                    "paginator": {
                        "type": "offset",
                        "limit": 100,
                        "offset_param": "offset",
                        "limit_param": "limit",
                        "total_path": "numFound",
                    },
                },
            },
        ],
    })


## 🔧 Step 3: Create the dlt Pipeline


In [ ]:
pipeline = dlt.pipeline(
    pipeline_name="ol_demo",
    destination="duckdb",
    dataset_name="ol_data",
    progress="log" # logs the pipeline run (Optiona)
)


## 🔍 Understanding the Pipeline

At this point we have two core building blocks:

- **source**: how data is fetched
- **pipeline**: destination, schema, and run metadata

Before running everything at once, we inspect each stage.

1. **Extract**: fetch raw data from the API  
2. **Normalize**: convert nested JSON to relational tables  
3. **Load**: write tables into DuckDB


![ETL Diagram](./images/etl_diagram.png)


Once these steps are clear, run the full workflow in one command:

```python
pipeline.run(source)
```


## ⬇️ Step 4: Extract


In [5]:
extract_info = pipeline.extract(openlibrary_source())


---

### What to print

After extraction, print a short summary:

- extracted **resources**
- resulting **tables**
- rows extracted per resource

This confirms extraction worked before normalization.


In [6]:
load_id = extract_info.loads_ids[-1]
m = extract_info.metrics[load_id][0]

print("Resources:", list(m["resource_metrics"].keys()))
print("Tables:", list(m["table_metrics"].keys()))
print("Load ID:", load_id)
print()

for resource, rm in m["resource_metrics"].items():
    print(f"Resource: {resource}")
    print(f"rows extracted: {rm.items_count}")
    print()


Resources: ['books']
Tables: ['books']
Load ID: 1770907406.962898

Resource: books
rows extracted: 3756



### What you should see after Extract

Typically you start with one resource (`books`),
and additional child tables appear during normalization.

So Extract validates data retrieval success.


## 🔄 Step 5: Normalize


In [7]:
normalize_info = pipeline.normalize()


In [8]:
load_id = normalize_info.loads_ids[-1]
m = normalize_info.metrics[load_id][0]

print("Load ID:", load_id)
print()

print("Tables created/updated:")
for table_name, tm in m["table_metrics"].items():
    # skip dlt internal tables to keep it beginner-friendly
    if table_name.startswith("_dlt"):
        continue
    print(f"  - {table_name}: {tm.items_count} rows")


Load ID: 1770907406.962898

Tables created/updated:
  - books: 3756 rows
  - books__author_key: 4600 rows
  - books__author_name: 4600 rows
  - books__ia: 3422 rows
  - books__ia_collection: 2724 rows
  - books__language: 3748 rows
  - books__id_standard_ebooks: 12 rows
  - books__id_librivox: 60 rows
  - books__id_project_gutenberg: 54 rows


### What happened during Normalize?

Running `pipeline.normalize()` splits nested fields into child tables.

Examples:
- `books`
- `books__author_name`
- `books__author_key`
- `books__language`
- `books__ia`

The key idea is transforming nested JSON into an analysis-friendly **relational schema**.


In [13]:
# Display schema
pipeline.default_schema


<dlt.Schema(name='rest_api', version=2, tables=['_dlt_version', '_dlt_loads', 'books', '_dlt_pipeline_state', 'books__author_key', 'books__author_name', 'books__ia', 'books__ia_collection', 'books__language', 'books__id_standard_ebooks', 'books__id_librivox', 'books__id_project_gutenberg'], version_hash='ZJIabaQJ9DAYgsR04wEVeXOgU80roBUfdvrR2YoBEyU=')>

## 📤 Step 6: Load


In [10]:
load_info = pipeline.load()


Normalized data is now loaded into DuckDB and ready for querying.


## 🚀 Step 7: Run the Full Pipeline


In [11]:
load_info = pipeline.run(openlibrary_source())


<h3>What does <code>pipeline.run()</code> do?</h3>

<p><code>pipeline.run(source)</code> runs these 3 stages in order:</p>

<ol>
  <li><strong>Extract</strong> – fetch data</li>
  <li><strong>Normalize</strong> – normalize schema</li>
  <li><strong>Load</strong> – load into the database</li>
</ol>


## 🔎 Step 8: Inspect the Loaded Data

Use `pipeline.dataset()` to inspect generated tables,
then preview rows from the main table.


In [12]:
ds = pipeline.dataset()


In [13]:
ds.tables


['books',
 'books__author_key',
 'books__author_name',
 'books__ia',
 'books__ia_collection',
 'books__language',
 'books__id_standard_ebooks',
 'books__id_librivox',
 'books__id_project_gutenberg',
 '_dlt_version',
 '_dlt_loads',
 '_dlt_pipeline_state']

In [17]:
df = ds.books.df()      # main table
df.head(3)


,cover_edition_key,cover_i,ebook_access,edition_count,first_publish_year,has_fulltext,key,lending_edition_s,lending_identifier_s,public_scan_b,title,_dlt_load_id,_dlt_id,subtitle
0,OL61027601M,15155833,borrowable,396,1997,True,/works/OL82563W,OL38565767M,harrypotterylapi0000rowl_q5r6,False,Harry Potter and the Philosopher's Stone,1770819876.9353185,lGJrV2BS8Z9qJQ,None
1,OL26378158M,15158660,printdisabled,144,2007,True,/works/OL82586W,None,None,False,Harry Potter and the Deathly Hallows,1770819876.9353185,F9W0WQlLwgvsFw,None
2,OL26234270M,10580435,borrowable,278,1999,True,/works/OL82536W,OL48101764M,bdrc-W8LS66814,False,Harry Potter and the Prisoner of Azkaban,1770819876.9353185,kSdfO1XbBVAjmQ,None


## 💡 Conclusion

dlt automates the following steps:

- API requests
- JSON normalization
- table creation
- database loading
- basic dataset inspection

This significantly reduces pipeline implementation effort.
